In [78]:
import pandas as pd
from sklearn.ensemble import IsolationForest

# Load the data
df = pd.read_csv('Dataset-anomalies.csv')

df.head()

cols_to_drop = ['S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR', 'Room_Occupancy_Count']
df.drop(cols_to_drop,axis=1,inplace=True)

df.head()


from datetime import datetime
from calendar import timegm

df['Timestamp'] = 0
df['Index'] = 0

for index, line in enumerate(df.values):
    date = line[0]
    time = line[1]
    finalDate = datetime.strptime(date + " " + time, '%Y/%m/%d %H:%M:%S')    
    timestamp = timegm(finalDate.timetuple())
    df.at[index, 'Timestamp'] = finalDate
    df.at[index, 'Index'] = index



import random
randomlist = [100, 1000, 1879, 3720, 2540, 5640, 6371, 7625, 9242]
# for i in range(0,5):
#     n = random.randint(0,len(df.index))
#     randomlist.append(n)
# print(randomlist)
# test1 = False
# for value in randomlist:
#     for index in range(value, value + 20):
#         if not test1:
#             df.at[index, 'S1_Temp'] =  random.uniform(35.1, 40.1)
#         else:
#             df.at[index, 'S1_Temp'] =  random.uniform(25.1, 35.1)
#     if not test1:
#         test1 = True
#     else:
#         test1 = False
previousNr = 0.0
numbersNR = []
import statistics
for index in range(1879, 1879 + 400):
    numbersNR.append(index)
    
for index, item in enumerate(numbersNR):
    if index < len(numbersNR) / 2:
        if previousNr == 0.0:
            previousNr = df.at[item, 'S1_Temp']
        df.at[item, 'S1_Temp'] = previousNr + 0.05
        previousNr = df.at[item, 'S1_Temp']
    else:
        df.at[item, 'S1_Temp'] = previousNr - 0.05
        previousNr = df.at[item, 'S1_Temp']

numbersNR = []
previousNr = 0.0
for index in range(6371, 6371 + 600):
    numbersNR.append(index)
    
for index, item in enumerate(numbersNR):
    if index < len(numbersNR) / 2:
        if previousNr == 0.0:
            previousNr = df.at[item, 'S1_Temp']
        df.at[item, 'S1_Temp'] = previousNr + 0.02
        previousNr = df.at[item, 'S1_Temp']
    else:
        df.at[item, 'S1_Temp'] = previousNr - 0.02
        previousNr = df.at[item, 'S1_Temp']

cols_to_drop = ['Date', 'Time']
df.drop(cols_to_drop,axis=1,inplace=True)

model = IsolationForest(contamination='auto', bootstrap=False, warm_start=False, random_state=42) # Contamination  is the "assumed" amount of outliers in the entire dataset
model.fit(df[['S1_Temp']])

df['outliers']=pd.Series(model.predict(df[['S1_Temp']])).apply(lambda x: 'yes' if (x == -1) else 'no' )
df.query('outliers=="yes"')

import plotly.express as px

fig = px.scatter(df.reset_index(), x='Index', y='S1_Temp', color='outliers',  hover_data='Timestamp')
fig.update_xaxes(
    rangeslider_visible=True,
)
fig.show()